# Avance 3: Modelo Baseline (Evolución y Evaluación del Sistema RAG)

# **Procesamiento de Lenguaje Natural**
## Maestría en Inteligencia Artificial Aplicada
### Tecnológico de Monterrey

* **Nombres y matrículas**
    * Sarmiento Cervantes Jacqueline: A01795863
    * Mayoral Terán Alexandro: A01795899
* **Número de equipo: 8**

**Propósito de este notebook:**  
Este documento es autocontenido y acompaña el Proyecto Integrador de la maestría. Su objetivo es presentar la evolución del motor de recuperación aumentada por generación (RAG) diseñado para apoyar la interpretación de documentos regulatorios y, posteriormente, la construcción de requerimientos de información financiera para la DISF.

El avance se concentra en la evaluación de métodos de recuperación de información. Para ello se comparan baselines léxicos, recuperación semántica con embeddings, búsqueda híbrida, expansión de consultas y reranking. La intención no es presentar un chatbot productivo, sino construir evidencia cuantitativa y cualitativa sobre qué estrategia recupera mejor los fragmentos normativos relevantes bajo restricciones de precisión, costo y latencia.

Las celdas de código importan directamente la arquitectura real del proyecto (`src/nlp_core`) para mostrar cómo se conectan los componentes implementados. Algunas operaciones costosas de ingesta o evaluación masiva se reportan a partir de artefactos ya generados en `data/03_output/evaluaciones`, con el fin de que el notebook sea revisable sin repetir llamadas innecesarias a servicios externos.

## 0.1 Evolución del Alcance y Prueba en Múltiples Dominios Regulatorios

El caso de uso original surge de una necesidad institucional: apoyar a analistas de Banco de México en la interpretación de normativa financiera y en el diseño de requerimientos de información. En los avances previos se trabajó principalmente con documentos relacionados con liquidez, como CCL, CFEN y liquidez intradía, porque es un dominio particular que trata de un tema en específico: el riesgo de liquidez.

Para este avance se amplió el conjunto de pruebas hacia documentos de crédito. Esta decisión no representa un cambio de objetivo del proyecto, sino una forma de evaluar si el motor de recuperación puede comportarse razonablemente en más de una familia normativa. En un contexto real de la DISF, un asistente de consulta regulatoria no debería depender de un único tema; debería poder recuperar evidencia útil en documentos con vocabularios, fórmulas, estructuras y criterios distintos.

Por esa razón, el avance 3 se interpreta como una prueba de generalización del pipeline RAG: se mantiene el propósito de apoyar la consulta y estructuración regulatoria, pero se incorpora crédito como un dominio adicional para construir un benchmark de recuperación más controlado y comparable.

## 0.2 Alineación con la Rúbrica Adaptada para LLM/RAG

La rúbrica del avance 3 solicita construir un modelo baseline y evaluar si métodos más complejos aportan valor medible. En un proyecto RAG, el concepto de “modelo” se traduce en estrategias de recuperación y ordenamiento de fragmentos normativos. Por ello, este notebook organiza la evidencia de la siguiente forma:

| Criterio | Interpretación en este proyecto | Evidencia en el notebook |
|---|---|---|
| BL1 | Definir baselines trivial y simple | BoW como baseline trivial; TF-IDF/BM25 como baselines simples; embeddings e híbridos como métodos avanzados |
| BL2 | Analizar contribución de componentes | Comparación entre recuperación léxica, semántica, híbrida, expansión de consultas y reranking |
| BL3 | Revisar memorización o contaminación | Discusión de prueba sin contexto y riesgo de que el LLM responda por conocimiento paramétrico |
| BL4 | Declarar métrica primaria a priori | NDCG@10 como métrica primaria; Recall@10, MAP@10 y latencia como métricas secundarias |
| BL5 | Definir desempeño mínimo y costo operativo | Comparación de mejora contra baselines y discusión de latencia/revisión humana |
| BL6 | Política de evaluación | Evaluación sobre un conjunto fijo de consultas; el corpus se mantiene común y se separan las preguntas evaluadas |
| BL7 | Reproducibilidad | Uso de artefactos versionados, CSVs de resultados, temperatura controlada en jueces LLM y base Chroma persistida |
| BL8 | Análisis de errores | Separación entre falla de retrieval, falla de generación y falla de formato/estructura |

In [8]:
# 0. Preparación del Entorno
# Se añade la raíz del proyecto al path para poder importar los módulos de 'src'
import sys
from pathlib import Path
import os
from dotenv import load_dotenv

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env")
print("✅ Entorno preparado. Importaciones locales de src/ listas.")

✅ Entorno preparado. Importaciones locales de src/ listas.


## 1. El Framework de Evaluación (Arena)

En este notebook usamos el término **Arena** como una metáfora de evaluación comparativa: un espacio controlado donde diferentes estrategias de recuperación “compiten” bajo las mismas consultas, el mismo conjunto de referencia y las mismas métricas. La finalidad no es declarar un ganador absoluto para todos los escenarios, sino observar qué método funciona mejor bajo distintas condiciones de precisión, costo, latencia y complejidad.

La Arena evita depender de ejemplos aislados o de validaciones “a ojo”. Cada método recibe las mismas preguntas y se evalúa contra el mismo conjunto de respuestas esperadas. Esto permite comparar un baseline léxico contra métodos semánticos y contra pipelines más complejos, como búsqueda híbrida, expansión de consultas y reranking.

### 1.1 El Archivo de Pruebas (Golden Dataset)

Se construyó un archivo `evaluacion_dataset.json` como conjunto de referencia inicial. Este archivo contiene 30 consultas representativas del dominio financiero-regulatorio y, para cada una, un fragmento esperado o texto clave que el sistema debería recuperar. En esta iteración, varias consultas se enfocan en crédito para probar el comportamiento del motor en conjunto con el dominio de liquidez trabajado inicialmente.

El conjunto de referencia todavía es perfectible: en una versión de producción debería ampliarse con más documentos, más dominios regulatorios y anotaciones revisadas por expertos. Sin embargo, para este avance cumple una función importante: permite medir de manera repetible si una estrategia de recuperación mejora o no frente a los baselines.

In [9]:
# 1.1 Cargando el Evaluador y el Golden Dataset real
from src.nlp_core.evals.evaluador import EvaluadorRAG
import json

ruta_dataset = project_root / "data" / "evaluacion_dataset.json"
evaluador = EvaluadorRAG(ground_truth_path=str(ruta_dataset))

# Mostramos el primer registro real del archivo de pruebas
ejemplo = evaluador.ground_truth[0] if hasattr(evaluador, 'ground_truth') and evaluador.ground_truth else {"Error": "Dataset no encontrado"}
print("Ejemplo Estructural del Golden Dataset (Query 0):")
print(json.dumps(ejemplo, indent=2, ensure_ascii=False)[:300] + "...\n}")

Ejemplo Estructural del Golden Dataset (Query 0):
{
  "query_id": "Q01",
  "pregunta": "¿Para consumo no revolvente cómo se define el pago realizado?",
  "dificultad": "Baja",
  "documentos_esperados": [
    {
      "archivo": "CUB_extracto.md",
      "jerarquia_esperada": "CNR",
      "texto_clave_esperado": "Monto correspondiente a la suma de los...
}


### 1.2 Declaración de Métricas A Priori (BL4) y Telemetría

Antes de comparar métodos, se declara **NDCG@10 como métrica primaria**. Esta decisión se toma porque en un sistema RAG no basta con recuperar el fragmento correcto en cualquier posición: mientras más arriba aparezca el contexto relevante, mayor probabilidad tiene el LLM generador de usarlo correctamente.

1. **NDCG@10 (*Normalized Discounted Cumulative Gain*)[métrica primaria]**  
    * *Cómo opera:* Aplica una atenuación logarítmica para penalizar si el documento correcto aparece muy abajo en el ranking.
    * *Ventajas:* En RAG, el orden estricto importa inmensamente. Obliga a que los fragmentos más relevantes queden en el Top 1-3, maximizando la "atención" del LLM generador.
    * *Desventajas:* Matemáticamente más complejo de interpretar que un simple porcentaje.

2. **Recall@10 (métrica secundaria)**  
    * *Cómo opera:* Mide qué porcentaje de las respuestas esperadas aparecieron en el Top 10.
    * *Ventajas:* Es vital porque si el texto legal no llega al prompt, el LLM final forzosamente alucinará (ya que no puede inventar leyes).
    * *Desventajas:* No le importa el orden; da la misma calificación si estaba en la posición 1 o en la 10.

3. **MAP@10 (métrica secundaria)**  
    * *Cómo opera:* Resume la precisión promedio considerando la posición del primer acierto. Complementa a NDCG y Recall.
    * *Ventajas:* Excelente para búsquedas binarias.
    * *Desventajas:* Inferior al NDCG para penalizar la posición estricta en rankings largos.

**Latencia y tokens de contexto (métricas de telemetría)**  
   Permiten analizar el costo de cada estrategia. Un pipeline puede tener mayor recall, pero ser menos viable si multiplica la latencia o el volumen de texto enviado al modelo.

### 1.3 Tres Modos de Auditoría

La evaluación de recuperación no es trivial porque el texto esperado puede aparecer con variaciones de redacción, fórmulas, encabezados o contexto parcial. Por ello se consideran tres formas de evaluación:

| Evaluador | Ventajas | Desventajas | Uso recomendado |
|---|---|---|---|
| `exact_match` | Determinístico, barato, reproducible | Muy estricto; genera falsos negativos si el texto fue parafraseado o fragmentado | Prueba rápida de regresión |
| Revisión humana | Máxima validez experta | Costosa, lenta y difícil de repetir semanalmente | Auditoría final o calibración de muestra |
| `llm_judge` | Escalable, reconoce paráfrasis y suficiencia semántica | No es verdad absoluta; puede sesgarse o ser inconsistente | Evaluación iterativa y comparación de métodos |

En este avance, el **LLM como juez** se usa como evaluador proxy para iterar rápidamente. No reemplaza la revisión humana, pero permite comparar múltiples pipelines bajo las mismas reglas sin que el equipo tenga que calificar manualmente cientos de resultados. Para una versión productiva, se recomienda calibrar el juez LLM con una muestra revisada por expertos.

### 1.4 Baselines y Métodos Comparados (BL1 y BL2)

Para que la comparación sea defendible, se separan los métodos en niveles de complejidad. Esto permite responder si las técnicas semánticas realmente aportan valor o si un buscador clásico sería suficiente.

| Método | Rol en la evaluación | Qué prueba | Ventaja | Limitación |
|---|---|---|---|---|
| BoW | Baseline trivial | Coincidencia simple de términos | Muy simple y barato | No pondera importancia ni entiende contexto |
| TF-IDF | Baseline simple | Ponderación por rareza de términos | Bueno para vocabulario regulatorio exacto | Falla con paráfrasis o sinónimos |
| BM25 | Baseline simple IR | Ranking léxico robusto | Estándar clásico en recuperación de información | No captura equivalencia semántica profunda |
| Embeddings | Método semántico | Similitud conceptual | Recupera fragmentos aunque cambie la redacción | Puede sufrir deriva semántica |
| Híbrido RRF | Método combinado | Fusión léxica + semántica | Reduce fallas individuales de BM25 y embeddings | Aumenta complejidad y dependencias |
| Cross-Encoder | Reranking | Reordena candidatos leyendo pregunta y fragmento juntos | Mejora el orden de resultados | Incrementa latencia |
| MultiQuery | Expansión de consulta | Genera paráfrasis de la pregunta | Mejora cobertura léxica y semántica | Puede introducir ruido |
| HyDE | Expansión semántica | Genera un documento hipotético para buscar por similitud | Conecta preguntas cortas con lenguaje normativo | Riesgo de orientar la búsqueda hacia una hipótesis incorrecta |

Bajo esta lógica, el proyecto no compara solamente “modelos”, sino **decisiones de arquitectura RAG**. El criterio central es si cada componente mejora la recuperación frente a los baselines y si esa mejora justifica su costo operativo.

### 1.5 `pipeline.py` como Orquestador del Experimento

El archivo `src/nlp_core/pipeline.py` funciona como un **orquestador**. Su responsabilidad no es almacenar documentos ni calcular todas las métricas, sino coordinar las etapas del flujo de recuperación:

1. recibe una consulta del usuario;
2. opcionalmente aplica expansión de consulta (`multi_query`, `hyde` o ambas);
3. selecciona el recuperador base (`bow`, `tfidf`, `bm25`, `embeddings` o `hibrido`);
4. acumula y deduplica documentos candidatos;
5. opcionalmente aplica reranking con Cross-Encoder;
6. devuelve los documentos finales ordenados.

Esta separación permite comparar estrategias sin reescribir todo el sistema. En términos de ingeniería, `pipeline.py` hace que la Arena sea modular: se puede activar o desactivar cada componente y medir su impacto sobre las mismas consultas.

### 1.6 Contribución de Componentes (BL2)

La rúbrica BL2 pide analizar la importancia de características o componentes, pero en un sistema RAG esto no se interpreta como importancia causal de variables tabulares. En este proyecto se analiza como una comparación **ablation-style** entre configuraciones del pipeline: se observa qué cambia al agregar ponderación léxica, recuperación semántica, fusión híbrida, expansión de consultas o reranking.

Estas comparaciones son **correlacionales, no causales**. Es decir, una mejora en NDCG@10 o Recall@10 asociada con un componente no demuestra por sí sola que ese componente sea la única causa de la mejora; también influyen el corpus, el chunking, la consulta, el juez de evaluación y la posición del fragmento relevante. Aun así, el análisis permite identificar qué decisiones arquitectónicas parecen aportar más valor y cuáles incrementan latencia o complejidad.

La contribución de componentes se revisa en cuatro dimensiones:

- **Retrieval con/sin señales semánticas:** comparación entre BM25/TF-IDF y embeddings.
- **Sensibilidad a `k`:** se reportan Recall@5 y Recall@10 para observar si el acierto aparece temprano o solo al ampliar el contexto.
- **Longitud y costo del contexto:** se monitorean tokens promedio y latencia.
- **Orden y calidad de candidatos:** se compara recuperación base contra variantes con Cross-Encoder y expansión de consultas.

El análisis de qué chunks aparecen con mayor frecuencia en Top-K queda como trabajo posterior, porque requiere una auditoría más detallada por `chunk_id` y por tipo de consulta. Sin embargo, la Arena ya deja la estructura necesaria para hacer esa revisión en futuras iteraciones.

## 2. Diagnóstico de la Línea Base: Pérdida de Contexto (*Context Loss*)

El primer paso consistió en establecer líneas base. Para BL1 se consideran dos niveles:

- **Baseline trivial:** BoW, que recupera por coincidencia simple de términos.
- **Baseline simple:** TF-IDF y BM25, que representan métodos clásicos de recuperación léxica más sólidos.

Estos métodos son importantes porque establecen el piso mínimo que una solución RAG debe superar. Si embeddings, búsqueda híbrida o reranking no mejoran de forma medible a BM25/TF-IDF, entonces el componente semántico no estaría justificando su costo.

Durante la evaluación inicial apareció un problema típico en documentos regulatorios extensos: la **pérdida de contexto**. Un fragmento puede contener una fórmula, un inciso o una definición, pero perder los encabezados que explican a qué cartera, indicador o capítulo pertenece. Cuando esto ocurre, el vector del fragmento queda semánticamente **“huérfano”** y la búsqueda puede fallar aunque el texto relevante exista en el corpus.

Un ejemplo específico sería imaginar que un Título Legal es "Anexo 33: Cartera Automotriz", y cinco incisos abajo el texto dice: "La fórmula de cálculo será X+Y". Al segmentar (hacer chunking), el vector matemático de este fragmento queda completamente ciego a que pertenece al mundo "Automotriz". La similitud de cosenos ante una pregunta falla irremediablemente.

> Nota ejecutable: la ingesta completa y la reconstrucción de ChromaDB pueden tardar varios minutos. Para mantener este notebook revisable, se muestran fragmentos conceptuales y se reutilizan artefactos persistidos en `data/03_output`.

In [ ]:
# Fragmento demostrativo de la estrategia inicial (No ejecutar, solo lectura)
'''
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [("#", "Título"), ("##", "Capítulo"), ("###", "Artículo")]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# Resultado trágico: Un fragmento hijo sin los nombres de sus secciones padre se vuelve invisible en el espacio vectorial.
'''
print("Fragmento de chunking conceptual omitido.")

## 3. Mitigación 1: Contextual Retrieval y Normalización L2
    
Para resolver la orfandad semántica, pasamos a un **Pipeline de Pre-procesamiento** (Interceptar el chunk y enriquecerlo *antes* de enviarlo al modelo de Embeddings).

### 3.1 Normalización L2 Explícita (Fundamento Matemático)
Antes de mejorar el texto, corregimos la base matemática. Se aplicó rigurosamente una **Normalización L2 Explícita** a los vectores. Esto equipara la Similitud Coseno al cálculo de Producto Punto, lo que permite acelerar masivamente ChromaDB en producción. Se descartó intencionalmente estandarizar post-vectorización (ej. StandardScaler) para no saturar memoria ni destruir los pesos originales IDF.

### 3.2 Inyector de Ruta vs Contextual Retrieval
- **Inyector de Metadatos:** Concatena los títulos al principio del fragmento (ej. `[Ruta: Anexo 33 > Cartera Automotriz] 
 La fórmula es...]`). Aumentó el Recall al **66.67%**. Es rápido, pero "ensucia" el documento con cadenas largas.
- **Contextual Retrieval (El Estado del Arte):** Se usa un LLM rápido (`gpt-4o-mini`) durante la ingesta. El LLM lee el documento entero y redacta una oración de contexto holístico que se incrusta en el texto.
- *Efectos:* Aumentó el Recall drásticamente al **73.33%**. Estas palabras empujan matemáticamente al vector resultante hacia su verdadero clúster financiero. *Desventaja:* Multiplica el costo monetario y tiempo durante la Ingesta.

> 💡 **Nota Ejecutable:** A continuación instanciamos el pipe apuntando a nuestra base de datos Chroma real (ya pre-procesada).

In [4]:
# 3.1 Instanciando el Motor de Búsqueda sobre nuestra BD Chroma real
from src.nlp_core.retrieval import MotorBusqueda
from src.nlp_core.pipeline import PipelineRecuperacion

# Nos conectamos a ChromaDB (ya contiene los embeddings con Contextual Retrieval inyectado)
motor = MotorBusqueda(
    persist_dir=project_root / "data" / "03_output" / "chroma_db",
    collection_name="regulacion_disf"
)

# Pipeline Base (Solo Embeddings Semánticos sin expansiones)
pipeline_base = PipelineRecuperacion(motor, documentos_raw=[], base_retriever="embeddings", query_expansion=None)

query_prueba = "¿Cómo se calcula la estimación preventiva de tarjetas de crédito?"
res_base = pipeline_base.invoke(query_prueba, k=1)

print("📝 Resultado Pipeline Base (Contextual Retrieval):")
print(res_base[0].page_content[:250] + "..." if res_base else "Sin resultados")

C:\Users\Alexandro\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Buscando: '¿Cómo se calcula la estimación preventiva de tarjetas de crédito?'...
📝 Resultado Pipeline Base (Contextual Retrieval):
[Contexto generado por IA: El fragmento se refiere a la estimación preventiva para riesgos crediticios, la cual está relacionada con la normativa establecida en el Anexo 33 del Banco de México, específicamente en el contexto de la contabilidad para i...


## 4. Mitigación 2: Expansión de Consultas (Multi-Query y HyDE)
    
A pesar de tener fragmentos impecables, descubrimos una **asimetría semántica**: Las leyes son formales; las consultas humanas son cortas y ambiguas.

### 4.1 Multi-Query (Múltiples Perspectivas)
- *Cómo opera:* El LLM genera múltiples paráfrasis de la pregunta original.
- *Ventaja:* Atrapa documentos con sinónimos no contemplados.
- *Desventaja:* Triplica la carga en la base de datos y trae ruido redundante.

### 4.2 HyDE (Hypothetical Document Embeddings)
- *Cómo opera:* El LLM imagina que es un experto y *redacta un artículo hipotético* respondiendo la pregunta. Vectorizamos esa respuesta alucinada para buscar.
- *Ventaja:* Matemáticamente es más fácil hacer match entre "Documento-Documento".
- *Desventaja:* Si el LLM alucina la teoría incorrecta, desatará una búsqueda hacia artículos tóxicos o irrelevantes.

> 💡 **Nota Ejecutable:** Ejecutamos HyDE en vivo conectando el pipeline.

In [5]:
# 4.1 Prueba de ejecución en vivo de HyDE
pipeline_hyde = PipelineRecuperacion(motor, documentos_raw=[], base_retriever="embeddings", query_expansion="hyde")

print(f"🔮 Ejecutando Pipeline con HyDE para: '{query_prueba}'")
# Internamente, llama a `_generar_hyde` conectándose a OpenAI para alucinar la respuesta y luego buscar
res_hyde = pipeline_hyde.invoke(query_prueba, k=1)

print("\n📝 Resultado Pipeline HyDE:")
print(res_hyde[0].page_content[:250] + "..." if res_hyde else "Sin resultados")

🔮 Ejecutando Pipeline con HyDE para: '¿Cómo se calcula la estimación preventiva de tarjetas de crédito?'
Generando documento hipotético (HyDE) para: '¿Cómo se calcula la estimación preventiva de tarjetas de crédito?'...
Buscando: 'La estimación preventiva de tarjetas de crédito se calculará mediante la aplicación de un modelo de riesgo crediticio que considere variables cuantitativas y cualitativas, incluyendo el historial de pagos del cliente, la relación entre el saldo de la tarjeta y el límite de crédito, así como el comportamiento de la cartera de crédito en segmentos similares. La metodología deberá incluir un análisis de la probabilidad de incumplimiento (PD), la pérdida dado el incumplimiento (LGD) y la exposición al incumplimiento (EAD). Adicionalmente, se requerirá la actualización periódica de los supuestos utilizados y la validación del modelo a través de pruebas de estrés y backtesting, a fin de asegurar la robustez de la estimación en función de las condiciones del entorno

## 5. Mitigación 3: Búsqueda Híbrida y Cross-Encoder Reranking
    
La suma de expansiones ensucia el contexto. Era vital un "filtro" final estricto.

### 5.1 Búsqueda Híbrida (BM25 + Embeddings + RRF)
Lanza un hilo de similitud semántica y otro de búsqueda de palabras exactas (BM25 Lexical). Aplica el algoritmo **RRF (Reciprocal Rank Fusion)** para mezclar resultados. Garantiza que la "deriva semántica" (semantic drift) provocada por HyDE no pierda de vista palabras clave estrictas (ej. un folio).

### 5.2 Cross-Encoder Reranking (El Juez Final)
Un *Reranker* (`cross-encoder/ms-marco-MiniLM-L-6-v2`) es una red neuronal que toma como entrada `[PREGUNTA + SEPARADOR + DOCUMENTO]` y procesa la relación de todas las palabras a la vez. Es el estándar en producción para ordenar listas sucias, pero impone una altísima latencia local (2 a 4 segundos extra).

> 💡 **Nota Ejecutable:** A continuación instanciamos el CrossEncoder. La celda tarda varios segundos en completarse comparada con el Pipeline Base.

In [6]:
# 5.1 Prueba de Reranking con Cross-Encoder (Súper RAG local)
# Nota: Activamos expansión 'ambos' (HyDE + MultiQuery) y luego podamos la basura con CrossEncoder

pipeline_super = PipelineRecuperacion(
    motor, 
    documentos_raw=[], 
    base_retriever="embeddings", 
    query_expansion="ambos", 
    post_processing="cross_encoder"
)

print(f"⚖️ Ejecutando Súper RAG (Reranking de múltiples expansiones)...\n⏳ Esto tomará varios segundos por el peso del Cross-Encoder...")
res_super = pipeline_super.invoke(query_prueba, k=1)

print("\n🏆 Top 1 Definitivo tras Cross-Encoder:")
print(res_super[0].page_content[:250] + "..." if res_super else "Sin resultados")

⚖️ Ejecutando Súper RAG (Reranking de múltiples expansiones)...
⏳ Esto tomará varios segundos por el peso del Cross-Encoder...
Generando variantes (Multi-Query) para: '¿Cómo se calcula la estimación preventiva de tarjetas de crédito?'...
Generando documento hipotético (HyDE) para: '¿Cómo se calcula la estimación preventiva de tarjetas de crédito?'...
Buscando: '¿Cómo se calcula la estimación preventiva de tarjetas de crédito?'...
Buscando: '¿Cuál es el método para determinar la estimación preventiva en tarjetas de crédito?'...
Buscando: '¿Qué procedimientos se utilizan para calcular la provisión preventiva de tarjetas de crédito?'...
Buscando: '¿Cómo se establece la reserva preventiva para tarjetas de crédito en las normativas financieras?'...
Buscando: 'La estimación preventiva de tarjetas de crédito se calculará mediante la aplicación de un modelo de riesgo crediticio que contemple variables cuantitativas y cualitativas, incluyendo el historial de pagos del deudor, la relación entre 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


🏆 Top 1 Definitivo tras Cross-Encoder:
[Contexto generado por IA: El fragmento se refiere a la obligación de las instituciones de crédito de revelar el saldo de la estimación preventiva para riesgos crediticios, desglosándolo según las metodologías de calificación de la cartera de crédito...


## 6. Análisis de Viabilidad, Data Contamination (BL3) y Errores (BL8)

### 6.1 Prueba de Contaminación de Datos (Data Contamination Check - BL3)

En un sistema RAG es importante distinguir entre dos escenarios: que el modelo responda porque el fragmento relevante fue recuperado, o que responda usando conocimiento paramétrico aprendido durante su entrenamiento. Esta distinción es especialmente relevante con normativa pública, porque algunos documentos podrían estar presentes total o parcialmente en datos de entrenamiento de modelos comerciales.

Por ello se considera una **prueba sin contexto (*no-context test*)**: formular preguntas al LLM `GPT-4o-mini` sin pasarle fragmentos recuperados y observar si responde con precisión suficiente. Si el modelo responde de forma vaga, incompleta o mezclando criterios de otras jurisdicciones, se fortalece la hipótesis de que la recuperación documental sí aporta valor. Esta prueba no elimina por completo el riesgo de contaminación, pero ayuda a documentarlo de forma explícita.

En la prueba realizada, el modelo alucinó severamente o mezcló normativas europeas (Basilea). Esto demuestra científicamente que el RAG aporta la mayoría de la certeza regulatoria y valida el esfuerzo.

### 6.2 Lectura Específica de BL3: Sub/Sobreajuste, Memorización y Contaminación

A continuación se presenta el análisis de conceptos como subajuste, sobreajuste y contaminación de datos, según la rúbrica BL3. En un proyecto RAG estos conceptos no se evalúan igual que en un modelo supervisado tradicional, porque no se entrena un clasificador con particiones clásicas de `train` y `test`. Por ello se interpretan de la siguiente manera:

| Elemento de BL3 | Interpretación en este proyecto RAG | Cómo se atiende en este avance |
|---|---|---|
| Subajuste | El recuperador es demasiado simple y no encuentra fragmentos relevantes aun cuando existen en el corpus | Comparación de BoW, TF-IDF y BM25 contra *embeddings* e híbridos |
| Sobreajuste | La estrategia funciona bien solo para consultas muy parecidas al *benchmark*, pero no generaliza a nuevas formulaciones | Uso de consultas con distinta dificultad y recomendación de ampliar con paráfrasis/*held-out queries* |
| *Held-out queries* | En RAG se separan consultas, no documentos; el corpus regulatorio permanece común | El benchmark de 30 consultas se trata como conjunto fijo inicial para comparar métodos |
| *Queries multi-chunk* | Algunas respuestas regulatorias pueden requerir varios fragmentos, fórmulas o encabezados | Se identifica como riesgo metodológico; requiere ampliar el ground truth a múltiples chunks relevantes por consulta |
| *No-context test* | Verifica si el LLM responde sin recuperación, lo que indicaría memorización o conocimiento paramétrico | Se discute como control para distinguir respuesta generada por memoria frente a respuesta sustentada en contexto recuperado |
| *Data contamination check* | Riesgo de que normativa pública haya aparecido en preentrenamiento del LLM | Se propone probar preguntas parafraseadas o sintéticas y comparar respuestas con/sin contexto |

En esta etapa no se reportan métricas de train vs validación porque no se realizó *fine-tuning* ni entrenamiento supervisado. El equivalente metodológico es comparar la robustez del retrieval ante distintas consultas y documentar explícitamente cuándo el sistema depende del contexto recuperado.

### 6.3 Costo de Revisión Humana a Escala (BL5)

La revisión humana sigue siendo el estándar de mayor confianza, sobre todo en dominios regulatorios. Sin embargo, revisar manualmente cada fragmento recuperado para cada combinación de método, consulta e hiperparámetro no es escalable en entregas semanales ni en operación cotidiana.

Auditar una salida manualmente en promedio toma ~5 minutos. El escalamiento requeriría múltiples SMEs (*Subject Matter Experts*). ¿Justifica el costo del LLM vs el método actual? Totalmente. El RAG hace el 90% del trabajo pesado de lectura y estructuración. El experto pasa de ser un "Creador desde cero" que pierde días leyendo, a un mero "Editor/Aprobador", generando un altísimo Retorno de Inversión (ROI) en horas-hombre.

Por eso se usa una estrategia mixta: `llm_judge` permite iterar y comparar métodos con rapidez, mientras que la revisión humana debe reservarse para calibrar una muestra, revisar errores críticos y validar resultados antes de cualquier uso productivo. En términos de negocio, el objetivo no es sustituir al experto, sino moverlo de una tarea de búsqueda exhaustiva hacia una tarea de validación y criterio.

### 6.4 Análisis Desagregado de Errores por Etapa (BL8)

Los errores se separan por etapa para que las mejoras futuras sean accionables:

1. **Falla de recuperación:** el fragmento relevante no aparece en el Top-K. La solución debe mejorar chunking, metadatos, embeddings, búsqueda híbrida o expansión de consulta.

   * En el presente experimento, el buscador no encontró el requerimiento regulatorio. El LLM alucina por ignorancia. Sin embargo, logró rectificarse llegando al 90% de Recall.
   
3. **Recuperación correcta, interpretación incorrecta:** el fragmento sí aparece, pero el LLM generador lo usa mal o lo interpreta de forma incompleta. La solución corresponde a prompts, instrucciones, modelos más robustos o validación experta.

   * En el experimento, el motor trajo el texto perfecto, pero el LLM lo malinterpretó matemáticamente. (Requiere *Prompt Engineering* o modelos más potentes como GPT-4o).

   
5. **Falla de formato o estructura:** la información recuperada es suficiente, pero la salida no respeta el formato esperado. Esta etapa se mitiga con contratos de datos, esquemas Pydantic y validaciones posteriores.

   * En el experimento dichas fallas se mitigaron mediante contratos de datos estrictos en Pydantic.

Esta taxonomía evita atribuir todos los problemas al “modelo” de forma genérica y permite decidir si el siguiente avance debe mejorar recuperación, generación o estructuración.

### 6.5 Ejemplos Cualitativos de Errores y Hallazgos (BL8)

Además de reportar métricas agregadas, se revisaron casos específicos de los archivos de resultados para identificar patrones de error. Esta revisión no sustituye una auditoría humana exhaustiva, pero ayuda a separar fallas de recuperación, fallas de evaluación y riesgos de generación.

| Tipo de caso BL8 | Ejemplo observado | Evidencia en resultados | Lectura |
|---|---|---|---|
| Retrieval falló en método léxico | Q02: fórmula de Probabilidad de Incumplimiento de crédito de nómina | BM25 no encontró hit; Embeddings sí recuperó el fragmento en rank 2 con juez LLM | La fórmula puede no recuperarse bien solo con coincidencia léxica; la señal semántica ayuda |
| Retrieval falló por vocabulario o dominio | Q26: definición de liquidez intradía según Comité de Basilea | BM25 no encontró hit; Embeddings sí recuperó el fragmento en rank 5 | La expansión a distintos dominios muestra diferencias de vocabulario entre consulta y documento |
| Ranking tardío | Q16: fórmula regulatoria del CCL | En una variante avanzada, el hit aparece hasta rank 8 | El fragmento relevante existe, pero llega tarde; por eso NDCG@10 es más informativa que solo Recall@10 |
| Evaluador exacto demasiado estricto | Q01-Q05 en Embeddings | Exact match marca miss, pero LLM judge marca hit en los primeros lugares | La subcadena esperada puede no aparecer literal aunque el fragmento sea suficiente |
| Riesgo de generación alucinada | No se evalúa end-to-end en esta libreta | El benchmark actual mide recuperación, no calidad final de respuesta generada | Debe evaluarse en una fase posterior con respuestas clasificadas como útil/parcial/incorrecta/alucinación |
| Riesgo de formato inválido | No se evalúa directamente en esta Arena | La salida estructurada pertenece al flujo de generación/formularios, no solo retrieval | Se debe revisar con contratos Pydantic y ejemplos de salida JSON en avances posteriores |

De esta revisión se desprenden tres observaciones cualitativas para el siguiente avance:

1. El error más importante a reducir en esta etapa es el **retrieval fallido**, porque sin contexto correcto el LLM no puede responder con fundamento normativo.
2. El uso de `exact_match` es útil como prueba rígida, pero subestima la relevancia semántica; por eso debe complementarse con juez LLM y revisión humana.
3. Para evaluar alucinación y validez de formato se requiere una evaluación end-to-end adicional, donde el sistema no solo recupere chunks, sino que genere respuestas o estructuras y estas sean clasificadas manualmente.

## 7. Resultados y Determinaciones Finales

Los resultados se presentan en dos niveles. Primero se cargan, cuando existen, los archivos CSV generados por la Arena (`ARENA_RESULTADOS_llm_judge.csv` y `ARENA_RESULTADOS_exact_match.csv`). Esto evita depender únicamente de tablas escritas manualmente. Después se muestra una tabla consolidada de interpretación que resume las familias de métodos y las decisiones arquitectónicas más relevantes.

In [1]:
from src.utils.resultados_avance3 import mostrar_resultados_arena, mostrar_analisis_componentes_bl2

# Las carpetas evaluaciones_contextual y chroma_db_contextual son copias equivalentes
# de evaluaciones y chroma_db para este avance; por claridad se usa la ruta principal.
df_llm, df_exact, resumen_interpretativo = mostrar_resultados_arena(project_root)


tabla_componentes_bl2 = mostrar_analisis_componentes_bl2(df_llm)


### Resultados de la Arena con LLM como juez

,estrategia,modo_evaluacion,Recall@5,Recall@10,MAP@10,NDCG@10,Latencia_Promedio_Segundos,Tokens_Contexto_Promedio
3,4_Embeddings,llm_judge,100.00,100.00,0.8678,0.9014,0.32,2189.7
5,6_Hibrido_CrossEncoder,llm_judge,96.67,96.67,0.8122,0.8503,9.38,2199.0
10,11_Ambos_Hibrido_CrossEncoder,llm_judge,93.33,100.00,0.7915,0.8413,17.88,2192.1
4,5_Hibrido_RRF,llm_judge,93.33,100.00,0.7854,0.8376,0.76,2219.6
6,7_Embeddings_CrossEncoder,llm_judge,96.67,100.00,0.7758,0.8304,8.56,2212.6
8,9_HyDE_Embeddings_CrossEncoder,llm_judge,93.33,100.00,0.7753,0.8300,10.90,2262.7
7,8_MultiQuery_Embeddings_CrossEncoder,llm_judge,96.67,100.00,0.7504,0.8110,10.95,2202.0
9,10_Ambos_Embeddings_CrossEncoder,llm_judge,93.33,100.00,0.7268,0.7925,14.29,2195.6
1,2_TF-IDF,llm_judge,90.00,96.67,0.6928,0.7605,0.85,2073.5
0,1_BoW,llm_judge,86.67,93.33,0.5944,0.6780,1.05,2166.7


### Resultados de la Arena con coincidencia exacta

,estrategia,modo_evaluacion,Recall@5,Recall@10,MAP@10,NDCG@10,Latencia_Promedio_Segundos,Tokens_Contexto_Promedio
8,9_HyDE_Embeddings_CrossEncoder,exact_match,40.00,40.00,0.4000,0.4000,8.86,231.4
3,4_Embeddings,exact_match,36.67,36.67,0.3667,0.3667,0.25,217.3
6,7_Embeddings_CrossEncoder,exact_match,36.67,36.67,0.3667,0.3667,5.61,217.3
7,8_MultiQuery_Embeddings_CrossEncoder,exact_match,36.67,36.67,0.3667,0.3667,7.85,217.3
5,6_Hibrido_CrossEncoder,exact_match,23.33,23.33,0.2333,0.2333,6.21,218.7
1,2_TF-IDF,exact_match,20.00,20.00,0.2000,0.2000,0.61,217.2
4,5_Hibrido_RRF,exact_match,20.00,20.00,0.2000,0.2000,0.71,220.3
0,1_BoW,exact_match,16.67,16.67,0.1667,0.1667,0.63,224.2
2,3_BM25,exact_match,16.67,16.67,0.1667,0.1667,0.43,225.2


### Resumen interpretativo por familia de metodos

,Familia,Metodos,Lectura,Riesgo
0,Baselines lexicos,"BoW, TF-IDF, BM25",Establecen el piso de comparacion. Son rapidos...,"Pueden fallar ante parafrasis, formulas fragme..."
1,Recuperacion semantica,Embeddings + ChromaDB,Captura similitud conceptual y suele mejorar c...,Puede recuperar fragmentos semanticamente cerc...
2,Busqueda hibrida,BM25 + Embeddings + RRF,Combina coincidencia exacta y similitud semant...,Incrementa complejidad y requiere calibrar pes...
3,Expansion y reranking,"MultiQuery, HyDE, Cross-Encoder",Puede mejorar cobertura y ordenamiento final d...,"Aumenta latencia, costo y riesgo de introducir..."


### Analisis ablation-style de componentes (BL2)

,Comparacion,Componente observado,Metodo base,Metodo comparado,Delta NDCG@10,Delta Recall@10,Delta latencia (s),Lectura
0,BoW -> TF-IDF,Ponderacion lexica,1_BoW,2_TF-IDF,0.0825,3.34,-0.1993,Evalua si ponderar terminos por rareza mejora ...
1,BM25 -> Embeddings,Senal semantica,3_BM25,4_Embeddings,0.2452,13.33,-0.1577,Evalua si la similitud conceptual aporta sobre...
2,Embeddings -> Hibrido RRF,Fusion lexica + semantica,4_Embeddings,5_Hibrido_RRF,-0.0638,0.00,0.4374,Evalua si combinar senales reduce fallas indiv...
3,Embeddings -> Embeddings + CrossEncoder,Reranking,4_Embeddings,7_Embeddings_CrossEncoder,-0.0710,0.00,8.2425,Evalua si reordenar candidatos mejora el ranki...
4,Embeddings + CrossEncoder -> MultiQuery + Cros...,Expansion de consulta,7_Embeddings_CrossEncoder,8_MultiQuery_Embeddings_CrossEncoder,-0.0194,0.00,2.3913,Evalua si parafrasear la consulta mejora cober...
5,Embeddings + CrossEncoder -> HyDE + CrossEncoder,Documento hipotetico,7_Embeddings_CrossEncoder,9_HyDE_Embeddings_CrossEncoder,-0.0004,0.00,2.3334,Evalua si buscar con una respuesta hipotetica ...


### 7.1 Lectura de los Resultados Observados

A partir de los archivos generados por la Arena, se observan dos lecturas complementarias:

1. **Con LLM como juez**, el método de embeddings presenta el mejor valor observado de NDCG@10 (`0.9014`) y Recall@10 (`100%`) dentro de los resultados guardados. Esto sugiere que, para el conjunto actual de preguntas, la recuperación semántica captura suficiente contexto normativo sin requerir necesariamente todas las capas avanzadas.

2. **Los métodos avanzados siguen siendo relevantes**, aunque no siempre ganen en la métrica primaria. Por ejemplo, las variantes con Cross-Encoder, HyDE o MultiQuery permiten explorar estrategias de alta precisión y analizar la frontera entre calidad, latencia y costo. Su valor no debe interpretarse únicamente como “ganar la tabla”, sino como herramientas disponibles para escenarios donde la consulta sea ambigua o el ranking inicial sea insuficiente.

3. **Con coincidencia exacta**, los valores son más bajos. Esto no necesariamente implica peor recuperación, sino que `exact_match` exige encontrar una subcadena específica. En documentos regulatorios, donde el contexto puede estar distribuido o redactado con variaciones, este evaluador tiende a producir falsos negativos.

4. **BM25, TF-IDF y BoW cumplen su función como baselines.** Aunque son menos sofisticados que embeddings o reranking, permiten establecer un piso claro. La comparación muestra si la complejidad adicional se justifica de forma medible.

En conjunto, estos resultados apoyan una arquitectura pragmática: mantener un modo base semántico por su buen balance entre calidad y latencia, y conservar métodos avanzados como opciones de auditoría o recuperación intensiva cuando el caso lo requiera.

## 8. Limitaciones, Criterio de Decisión y Siguientes Pasos

### 8.1 Limitaciones del Benchmark

El conjunto de evaluación utilizado en este avance permite comparar métodos de recuperación de forma repetible, pero todavía debe interpretarse como un benchmark inicial. Sus principales limitaciones son:

- El número de consultas es reducido frente a la diversidad real de preguntas que pueden surgir en la DISF.
- La evaluación debe ampliarse con más preguntas de liquidez, crédito, formularios, catálogos, validaciones y criterios de reporte.
- El juez LLM permite evaluar de forma escalable, pero no sustituye la validación de expertos. En una siguiente iteración debe calibrarse con una muestra revisada manualmente.
- La coincidencia exacta tiende a subestimar la recuperación cuando el fragmento relevante aparece con variaciones de redacción, fórmulas separadas o contexto distribuido en varios chunks.

Estas limitaciones no invalidan la comparación; delimitan el alcance de la evidencia obtenida y señalan qué debe fortalecerse en avances posteriores.

### 8.2 Criterio de Decisión para Métodos Avanzados

Un método avanzado se considera útil si mejora la métrica primaria, **NDCG@10**, o una métrica secundaria crítica como **Recall@10**, frente a los baselines simples, sin introducir una latencia desproporcionada para el caso de uso. En otras palabras, la selección no debe basarse únicamente en el mayor puntaje, sino en el balance entre relevancia, costo, latencia y facilidad de operación.

Bajo este criterio, un pipeline semántico rápido puede ser preferible para consultas frecuentes, mientras que estrategias como Cross-Encoder, HyDE o MultiQuery pueden reservarse para escenarios de auditoría, consultas ambiguas o casos donde el costo de no recuperar el fragmento correcto sea alto.

### 8.3 Por Qué No Priorizar *Fine-Tuning* en Esta Etapa

Aunque el *fine-tuning* contrastivo de *embeddings* podría ser una línea futura, no se prioriza en este avance. Antes de ajustar modelos, es necesario contar con un *benchmark* estable, una taxonomía de errores y evidencia clara de qué parte del sistema falla: recuperación, generación o estructuración. Aplicar *fine-tuning* sin esa base podría ocultar problemas de datos, chunking, metadatos o evaluación.

Por ello, este avance se concentra en establecer *baselines*, comparar configuraciones de recuperación y construir una metodología de evaluación reproducible. Una vez consolidado el conjunto de consultas y revisado por expertos, tendría más sentido evaluar si el ajuste de embeddings aporta una mejora adicional.


### 8.4 Reproducibilidad y Controles BL7

La reproducibilidad en un sistema LLM/RAG requiere más controles que en un notebook tradicional, porque algunas llamadas dependen de servicios externos, modelos alojados y prompts que pueden cambiar. Para este avance se documentan los siguientes controles:

| Elemento BL7 | Decisión en este proyecto | Estado actual |
|---|---|---|
| Temperatura / semilla | Las evaluaciones con juez LLM usan temperatura baja o cero cuando el componente lo permite, para reducir variabilidad entre ejecuciones | Implementado en el juez LLM (`temperature=0.0`) |
| Snapshot del modelo | Se registra el modelo usado para embeddings y jueces/generadores cuando aparece en el código | Parcial; se usa `text-embedding-3-small`, `gpt-4o-mini` y `gpt-4o` en componentes específicos |
| Versionado de prompts | Los prompts de HyDE, MultiQuery y juez LLM deben tratarse como artefactos del experimento | Parcial; actualmente viven en `pipeline.py` y `evaluador.py`; en una siguiente iteración conviene moverlos a archivos versionados |
| Logs estructurados | Para auditar una corrida se requieren consulta, respuesta, latencia, tokens, estrategia y modelo | Parcial; existen CSVs de evaluación y telemetría de tokens/latencia; las llamadas generativas podrían registrarse con mayor detalle |
| Cache de embeddings | La base vectorial debe construirse una vez y reutilizarse para evitar variación, costo y tiempo de cómputo | Implementado mediante ChromaDB persistido en `data/03_output` |
| Artefactos de evaluación | Los resultados deben guardarse fuera del output visual del notebook | Implementado mediante CSVs en `data/03_output/evaluaciones` |
| Conjunto fijo de consultas | Las estrategias deben compararse contra las mismas preguntas | Implementado mediante `data/evaluacion_dataset.json` |

Estos controles permiten rehacer la comparación sin cambiar la regla de evaluación. Aun así, quedan mejoras claras para una versión posterior: separar los prompts a archivos con versión, registrar el nombre exacto del modelo en cada fila de resultados, guardar logs JSONL por llamada y documentar hashes de configuraciones relevantes.

### Conclusiones Institucionales y Técnicas

1. **La evaluación comparativa es el principal avance de esta entrega.**  
   La Arena permite comparar métodos bajo las mismas consultas, métricas y criterios. Esto transforma el proyecto de una demostración funcional a un experimento medible.

2. **Los baselines son indispensables.**  
   BoW, TF-IDF y BM25 no son métodos “menores”; son el punto de referencia que justifica o cuestiona el uso de arquitecturas más complejas. Si una estrategia semántica no supera a estos métodos, no debería adoptarse solo por ser más moderna.

3. **La expansión a crédito fortalece la prueba de generalización.**  
   Incorporar consultas de crédito permite observar el comportamiento del pipeline en un dominio regulatorio adicional al de liquidez. El objetivo institucional sigue siendo construir un asistente de consulta y apoyo regulatorio aplicable a múltiples familias normativas.

4. **El LLM como juez es útil, pero no definitivo.**  
   Su principal ventaja es permitir iteración rápida y evaluación semántica. Su limitación es que sigue siendo un proxy. En producción, debe calibrarse con revisión humana experta.

5. **Existe una frontera entre precisión y costo operativo.**  
   Métodos como Cross-Encoder, MultiQuery o HyDE pueden mejorar ciertos escenarios, pero aumentan latencia y complejidad. Por ello, una arquitectura viable debería distinguir entre un modo rápido de consulta y un modo de alta precisión para auditorías o preguntas críticas.

6. **Siguiente paso.**  
   Llegar al objetivo principal que es el diseño de formularios; revisar una muestra con expertos; y mantener el mismo benchmark para comparar avances posteriores sin cambiar la regla de medición.